# 🎰 Smart Betting : La Stratégie Mathématique (V2)

Ce notebook implémente l'alternative **"Smart Betting"**. Plutôt que de booster artificiellement des segments (comme Syndicate), cette stratégie :
1.  Calcule une probabilité précise avec la "Formule" (Régression Logistique).
2.  Divise la population en **Clusters Comportementaux** (Pays + Tranche d'Âge).
3.  Optimise mathématiquement le **Seuil de Classification** pour chaque cluster afin de maximiser le F1-Score localement.

C'est l'approche "Pure Math" (F1 Training: 0.7725).

---

In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import f1_score

print("📥 Loading Data...")
train_df = pd.read_csv('conversion_data_train.csv')
test_df = pd.read_csv('conversion_data_test.csv')

# Feature Engineering (Equation Style)
# On recrée le feature space "physique" du dataset
full_df = pd.concat([train_df.drop('converted', axis=1), test_df], axis=0).reset_index(drop=True)
full_df['pages_sq'] = full_df['total_pages_visited'] ** 2
full_df['age_sq'] = full_df['age'] ** 2
full_df['interaction'] = full_df['total_pages_visited'] * full_df['age']
full_df['rate_concept'] = full_df['total_pages_visited'] / (full_df['age'] + 1)

# Define Clusters (The "Betting Markets")
# Country + AgeBin (Young/Mid/Sen)
full_df['age_bin'] = pd.cut(full_df['age'], bins=[0, 25, 40, 100], labels=['Young', 'Mid', 'Sen'])
full_df['cluster'] = full_df['country'] + "_" + full_df['age_bin'].astype(str)

print(f"📊 Defined {full_df['cluster'].nunique()} Betting Markets.")

## 1. Entraînement de la Formule (Probabilités)

In [ ]:
numeric_features = ['age', 'total_pages_visited', 'pages_sq', 'age_sq', 'interaction', 'rate_concept']
categorical_features = ['country', 'source', 'new_user']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), categorical_features)
    ]
)

# Strong Logistic Regression (Reverse Engineered Model)
model = LogisticRegression(solver='lbfgs', max_iter=2000, C=1e9)
pipeline = Pipeline([('preprocessor', preprocessor), ('model', model)])

print("⚙️ Training Global Model...")
train_len = len(train_df)
X_train = full_df.iloc[:train_len]
y_train = train_df['converted']
X_test = full_df.iloc[train_len:]

pipeline.fit(X_train, y_train)
train_probs = pipeline.predict_proba(X_train)[:, 1]
test_probs = pipeline.predict_proba(X_test)[:, 1]
print("✅ Probabilities Computed.")

## 2. Optimisation des Seuils par Cluster

In [ ]:
print("🎯 Optimizing Thresholds per Market...")
clusters = full_df['cluster'].unique()
cluster_thresholds = {}
global_preds_train = np.zeros(train_len)

for c in clusters:
    mask_train = (full_df.iloc[:train_len]['cluster'] == c).values
    
    if mask_train.sum() < 50:
        cluster_thresholds[c] = 0.5
        continue
        
    y_c = y_train[mask_train]
    prob_c = train_probs[mask_train]
    
    best_t = 0.5
    best_f1 = 0
    
    if y_c.sum() == 0:
        best_t = 0.99 # No conversions seen -> Conservative
    else:
        for t in np.linspace(0.1, 0.9, 81):
            score = f1_score(y_c, (prob_c >= t).astype(int))
            if score > best_f1:
                best_f1 = score
                best_t = t
    
    cluster_thresholds[c] = best_t
    global_preds_train[mask_train] = (prob_c >= best_t).astype(int)
    
    print(f"   Category '{c}': Best Threshold = {best_t:.2f} (F1: {best_f1:.3f})")

print(f"\n📊 Global Optimized Train F1: {f1_score(y_train, global_preds_train):.5f}")

## 3. Génération de la Soumission

In [ ]:
print("🔮 Placing Final Bets on Test Set...")
test_preds = np.zeros(len(test_df), dtype=int)

for c in clusters:
    mask_test = (full_df.iloc[train_len:]['cluster'] == c).values
    if mask_test.sum() == 0: continue
    
    thresh = cluster_thresholds.get(c, 0.5)
    prob_c = test_probs[mask_test]
    test_preds[mask_test] = (prob_c >= thresh).astype(int)

sub = pd.DataFrame({'converted': test_preds})
filename = 'submission_SMART_BETTING_GENERATED_V2.csv'
sub.to_csv(filename, index=False)
print(f"✅ Generated '{filename}' with {test_preds.sum()} conversions.")